# Image Captioning sur MS-COCO : préparation des données (phase 1)

Groupe 19, technique assignée : **Self-RAG**. Ce notebook réalise la phase « exploration des données et prétraitement » de l'énoncé (section 4.4.3) et produit les artefacts utilisés ensuite : splits figés, métadonnées d'objets, embeddings CLIP et index FAISS.

**Exécution.** Google Colab avec GPU (T4), cellules dans l'ordre. Les résultats sont écrits dans le dossier Drive `captioning`. Les images (environ 1,8 Go) sont retéléchargées dans `/content` à chaque session.

| Étape | Contenu | Sections de l'énoncé |
|---|---|---|
| 1 | Configuration et annotations COCO | 4.1 |
| 2 | Création des splits et anti-fuite | 4.2, 4.4.3 (2) |
| 3 | Téléchargement des images | 4.4.3 |
| 4 | Audit et nettoyage des données | 4.4.3 (1) |
| 5 | Extraction d'objets (YOLOv8m) et métadonnées | 4.4.3 (5) |
| 6 | Exploration n°1 (avant prétraitement) | 4.3 |
| 7 | Prétraitement des légendes | 4.4.3 (4) |
| 8 | Prétraitement des images | 4.4.3 (3) |
| 9 | Exploration n°2 (après prétraitement) | 4.3 |
| 10 | Embeddings image et texte (CLIP) | 4.4.3 (6) |
| 11 | Base de connaissances : index FAISS | 4.4.3 (7), 8 |
| 12 | Audit de fuite et contrôle qualité | 4.4.3 (2, 8) |

## 1. Configuration et annotations COCO

On monte Drive, on fixe le seed (42) et on charge les annotations officielles COCO 2017 (légendes de `train2017` et `val2017`). La fonction `sync_drive` force l'écriture sur Drive : sans elle, des fichiers peuvent ne pas être enregistrés si la session s'arrête.

In [ ]:
!pip install -q ultralytics faiss-cpu

import os, re, json, time, random, hashlib, subprocess
import numpy as np, pandas as pd
import requests, torch, transformers
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

SEED = 42
CAP = '/content/drive/MyDrive/captioning'          # dossier de sortie (Drive)
FIG, EMB, IDX = f'{CAP}/figures', f'{CAP}/embeddings', f'{CAP}/index'
for d in (CAP, f'{CAP}/splits', FIG, EMB, IDX):
    os.makedirs(d, exist_ok=True)
random.seed(SEED)

def sync_drive():
    """Force l'écriture sur Drive, puis remonte le disque."""
    drive.flush_and_unmount()
    drive.mount('/content/drive')

In [ ]:
subprocess.run("wget -nc -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P /content "
               "&& unzip -q -o /content/annotations_trainval2017.zip -d /content", shell=True, check=True)

with open('/content/annotations/captions_train2017.json') as f:
    cap_train = json.load(f)
with open('/content/annotations/captions_val2017.json') as f:
    cap_val = json.load(f)

train_info = {im['id']: im for im in cap_train['images']}
val_info   = {im['id']: im for im in cap_val['images']}

caps_by_img = defaultdict(list)
for a in cap_train['annotations'] + cap_val['annotations']:
    caps_by_img[a['image_id']].append(a['caption'].strip())

print("train2017 :", len(cap_train['images']), "images,", len(cap_train['annotations']), "légendes")
print("val2017   :", len(cap_val['images']), "images")

## 2. Création des splits et anti-fuite

- **Base de connaissances (train)** : 10 000 images de `train2017`.
- **Validation** : 100 images de `train2017`, tirées dans la même permutation, donc disjointes du train.
- **Test** : 1 000 images de `val2017` (les 100 premières forment le sous-ensemble `test100`, gardé pour comparaison).

Les splits sont **figés** : s'ils existent déjà dans `splits.json`, ils sont rechargés et jamais régénérés.

In [ ]:
SPLITS_PATH = f'{CAP}/splits/splits.json'
if os.path.exists(SPLITS_PATH):
    splits = json.load(open(SPLITS_PATH))
    print("splits existants rechargés (ils ne sont jamais régénérés)")
else:
    pool = [im['id'] for im in cap_train['images']]
    random.shuffle(pool)
    test_pool = [im['id'] for im in cap_val['images']]
    test_sel = random.sample(test_pool, 1000)
    splits = {"train": pool[:10000], "val": pool[10000:10100],
              "test1000": test_sel, "test100": test_sel[:100]}
    json.dump(splits, open(SPLITS_PATH, 'w'))
    print("splits créés et sauvegardés")

train_ids, val_ids, test_ids = splits['train'], splits['val'], splits['test1000']
assert not (set(train_ids) & set(val_ids)), "fuite : image commune au train et à la validation"
assert (len(train_ids), len(val_ids), len(test_ids)) == (10000, 100, 1000)

# (split, ids, annotations de référence) : utilisé par toutes les étapes suivantes
SPLIT_INFO = [("train", train_ids, train_info),
              ("val",   val_ids,   train_info),
              ("test",  test_ids,  val_info)]
print({s: len(i) for s, i, _ in SPLIT_INFO})

## 3. Téléchargement des images

Seules les 11 100 images des splits sont téléchargées (pas les archives complètes de plusieurs Go). Les fichiers déjà présents sont ignorés, donc la cellule peut être relancée.

In [ ]:
def download(args):
    url, path = args
    if os.path.exists(path):
        return True
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        with open(path, 'wb') as f:
            f.write(r.content)
        return True
    except Exception:
        return False

def download_split(ids, info, folder):
    os.makedirs(folder, exist_ok=True)
    jobs = [(info[i]['coco_url'], f"{folder}/{info[i]['file_name']}") for i in ids]
    with ThreadPoolExecutor(max_workers=16) as ex:
        results = list(tqdm(ex.map(download, jobs), total=len(jobs)))
    print(folder, ":", sum(results), "/", len(jobs), "ok")

for split, ids, info in SPLIT_INFO:
    download_split(ids, info, f'/content/images/{split}')

## 4. Audit et nettoyage des données

Contrôles de l'énoncé (4.4.3, étape 1) : images manquantes, corrompues, en niveaux de gris, doublons (empreinte MD5, y compris entre splits) et légendes (5 par image, non vides). Le rapport est sauvegardé dans `cleaning_report.json`.

In [ ]:
report = {"missing": 0, "corrupted": 0, "grayscale": 0}
hashes = defaultdict(list)

for split, ids, info in SPLIT_INFO:
    for i in tqdm(ids, desc=split):
        path = f"/content/images/{split}/{info[i]['file_name']}"
        if not os.path.exists(path):
            report["missing"] += 1
            continue
        try:
            with Image.open(path) as img:
                img.verify()                      # détecte les fichiers corrompus
            with Image.open(path) as img:
                if img.mode != "RGB":
                    report["grayscale"] += 1
        except Exception:
            report["corrupted"] += 1
            continue
        with open(path, 'rb') as f:
            hashes[hashlib.md5(f.read()).hexdigest()].append((split, i))

all_ids = set(train_ids) | set(val_ids) | set(test_ids)
cleaning_report = {
    **report,
    "duplicate_groups": sum(1 for v in hashes.values() if len(v) > 1),
    "duplicates_across_splits": sum(1 for v in hashes.values() if len({s for s, _ in v}) > 1),
    "bad_captions": sum(1 for i in all_ids
                        if len(caps_by_img[i]) < 5 or any(len(c) < 3 for c in caps_by_img[i])),
    "ids_overlap_train_val": len(set(train_ids) & set(val_ids)),
}
json.dump(cleaning_report, open(f'{CAP}/cleaning_report.json', 'w'))
print(cleaning_report)

## 5. Extraction d'objets (YOLOv8m) et métadonnées

Les objets sont extraits **directement des images** avec YOLOv8m (80 catégories COCO), au seuil de confiance 0,5. Pour chaque image on conserve les légendes brutes et la liste des objets (catégorie, confiance, boîte englobante). La version du détecteur et le seuil sont notés dans `detector_info.json`. Durée : environ 6 minutes sur GPU T4.

In [ ]:
import ultralytics
from ultralytics import YOLO

model = YOLO('yolov8m.pt')
CONF_THRESHOLD = 0.5

def extract_objects(ids, info, folder, batch_size=32):
    res = {}
    paths = [f"{folder}/{info[i]['file_name']}" for i in ids]
    for s in tqdm(range(0, len(paths), batch_size)):
        preds = model.predict(paths[s:s+batch_size], conf=CONF_THRESHOLD, verbose=False)
        for img_id, p in zip(ids[s:s+batch_size], preds):
            res[img_id] = [{"category": model.names[int(b.cls)],
                            "confidence": round(float(b.conf), 3),
                            "bbox": [round(float(x), 1) for x in b.xyxy[0].tolist()]}
                           for b in p.boxes]
    return res

objects = {}
for split, ids, info in SPLIT_INFO:
    objects.update(extract_objects(ids, info, f'/content/images/{split}'))

metadata = []
for split, ids, info in SPLIT_INFO:
    for i in ids:
        objs = objects[i]
        metadata.append({
            "image_id": i, "split": split, "file_name": info[i]['file_name'],
            "captions": caps_by_img[i], "objects": objs, "num_objects": len(objs),
            "categories": sorted(set(o['category'] for o in objs)),
        })
df = pd.DataFrame(metadata)

# Cohérence avec les splits figés
for split, ids, _ in SPLIT_INFO:
    assert set(df[df['split'] == split]['image_id']) == set(ids), f"métadonnées incohérentes ({split})"

json.dump({str(k): v for k, v in objects.items()}, open(f'{CAP}/objects.json', 'w'))
json.dump(metadata, open(f'{CAP}/metadata.json', 'w'))
json.dump({"detecteur": "YOLOv8m (yolov8m.pt)", "ultralytics": ultralytics.__version__,
           "seuil_confiance": CONF_THRESHOLD}, open(f'{CAP}/detector_info.json', 'w'), indent=2)
print("images sans objet :", int((df['num_objects'] == 0).sum()))
sync_drive()

## 6. Exploration n°1 (avant prétraitement)

Statistiques de l'énoncé (4.3) calculées à partir des objets extraits : distribution du nombre d'objets par image pour chaque split, top 50 des catégories, longueur des légendes et confiance des détections. Les figures et le résumé chiffré (`explo1.json`) sont sauvegardés.

In [ ]:
df['caption_len'] = df['captions'].apply(lambda cs: sum(len(c.split()) for c in cs) / len(cs))

print(df['split'].value_counts())
print(df.groupby('split')['num_objects'].describe())
print("Images sans objet :", int((df['num_objects'] == 0).sum()),
      f"({(df['num_objects'] == 0).mean()*100:.1f} %)")

# Nombre d'objets par image
fig, ax = plt.subplots(figsize=(8, 4))
for split in ['train', 'val', 'test']:
    ax.hist(df[df['split'] == split]['num_objects'], bins=range(0, 41),
            alpha=0.5, label=split, density=True)
ax.set_xlabel("Nombre d'objets par image")
ax.set_ylabel("Proportion")
ax.set_title("Distribution du nombre d'objets (avant prétraitement)")
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG}/explo1_num_objects.png', dpi=150)
plt.show()

# Top 50 des catégories
cat_counter = Counter(o['category'] for objs in df['objects'] for o in objs)
labels, counts = zip(*cat_counter.most_common(50))
fig, ax = plt.subplots(figsize=(9, 11))
ax.barh(labels[::-1], counts[::-1])
ax.set_xlabel("Nombre d'occurrences")
ax.set_title("Top 50 des catégories d'objets détectées (avant prétraitement)")
plt.tight_layout()
plt.savefig(f'{FIG}/explo1_top_categories.png', dpi=150)
plt.show()
print("catégories distinctes détectées :", len(cat_counter), "/ 80")

# Longueur des légendes et confiance des détections
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['caption_len'], bins=30)
axes[0].set_xlabel("Longueur moyenne des légendes (mots)")
axes[0].set_title("Longueur des légendes")
axes[1].hist([o['confidence'] for objs in df['objects'] for o in objs], bins=30)
axes[1].set_xlabel("Score de confiance")
axes[1].set_title("Confiance des détections")
plt.tight_layout()
plt.savefig(f'{FIG}/explo1_captions_conf.png', dpi=150)
plt.show()

explo1 = {
    "n_images": int(len(df)),
    "images_par_split": df['split'].value_counts().to_dict(),
    "images_sans_objet": int((df['num_objects'] == 0).sum()),
    "objets_total": int(sum(cat_counter.values())),
    "objets_par_image_moyenne": round(float(df['num_objects'].mean()), 2),
    "categories_distinctes": len(cat_counter),
    "top50_categories": cat_counter.most_common(50),
    "longueur_moyenne_legende": round(float(df['caption_len'].mean()), 2),
}
json.dump(explo1, open(f'{CAP}/explo1.json', 'w'), indent=2)
print({k: v for k, v in explo1.items() if k != 'top50_categories'})

## 7. Prétraitement des légendes

Les légendes originales sont **conservées intactes** (`captions`, utilisées pour l'évaluation). Pour l'indexation uniquement, on crée `captions_clean` : minuscules, ponctuation retirée, espaces normalisés. Les boîtes d'objets sont aussi données en coordonnées normalisées (`bbox_norm`, entre 0 et 1), car les images seront redimensionnées. Les images sans objet sont gardées et marquées (`no_object`) pour respecter les effectifs demandés.

In [ ]:
def clean_caption(c):
    c = c.lower().strip()
    c = re.sub(r"[^a-z0-9\s']", " ", c)      # supprime la ponctuation
    c = re.sub(r"\s+", " ", c).strip()        # espaces multiples
    return c

df['captions_clean'] = df['captions'].apply(lambda cs: [clean_caption(c) for c in cs])
df['caption_len_clean'] = df['captions_clean'].apply(lambda cs: sum(len(c.split()) for c in cs) / len(cs))
df['no_object'] = df['num_objects'] == 0

def get_info(row):
    return val_info[row['image_id']] if row['split'] == 'test' else train_info[row['image_id']]

def add_norm_bbox(row):
    inf = get_info(row)
    w, h = inf['width'], inf['height']
    new = []
    for o in row['objects']:
        x1, y1, x2, y2 = o['bbox']
        o2 = dict(o)
        o2['bbox_norm'] = [round(x1/w, 4), round(y1/h, 4), round(x2/w, 4), round(y2/h, 4)]
        new.append(o2)
    return new

df['objects'] = df.apply(add_norm_bbox, axis=1)
df.to_json(f'{CAP}/metadata_clean.json', orient='records')
print(df[['captions', 'captions_clean']].iloc[0].to_dict())
sync_drive()

## 8. Prétraitement des images

Correction de l'orientation EXIF, conversion en RGB (les images en niveaux de gris sont converties) et redimensionnement en 224×224 pour l'encodeur CLIP. Les images d'origine ne sont pas modifiées : le résultat est écrit dans `/content/images_clean`. Le nombre d'images corrigées par EXIF est compté pour le rapport.

In [ ]:
def preprocess_images(split, ids, info):
    out_dir = f'/content/images_clean/{split}'
    os.makedirs(out_dir, exist_ok=True)
    n_rotated = 0
    for i in tqdm(ids, desc=split):
        src = f"/content/images/{split}/{info[i]['file_name']}"
        dst = f"{out_dir}/{info[i]['file_name']}"
        with Image.open(src) as img:
            if img.getexif().get(274, 1) != 1:        # 274 = orientation EXIF
                n_rotated += 1
            img = ImageOps.exif_transpose(img).convert('RGB')
            img = img.resize((224, 224), Image.BICUBIC)
            img.save(dst, quality=95)
    return n_rotated

exif_counts = {split: preprocess_images(split, ids, info) for split, ids, info in SPLIT_INFO}
print("images tournées par EXIF :", exif_counts)

## 9. Exploration n°2 (après prétraitement)

Mêmes indicateurs avant et après : taille du vocabulaire, longueur des légendes, images en niveaux de gris, rotation EXIF, tailles d'image. Le tableau est sauvegardé dans `explo_comparison.csv`.

In [ ]:
modes, sizes = Counter(), Counter()
for split in ['train', 'val', 'test']:
    folder = f'/content/images_clean/{split}'
    for f in os.listdir(folder):
        with Image.open(f'{folder}/{f}') as img:
            modes[img.mode] += 1
            sizes[img.size] += 1
print(modes, sizes)

vocab_before = set(w for cs in df['captions'] for c in cs for w in c.split())
vocab_after  = set(w for cs in df['captions_clean'] for c in cs for w in c.split())
n_exif = sum(exif_counts.values())

comparison = pd.DataFrame({
    "Avant prétraitement": {
        "Taille du vocabulaire": len(vocab_before),
        "Longueur moyenne des légendes (mots)": round(df['caption_len'].mean(), 2),
        "Images en niveaux de gris": cleaning_report['grayscale'],
        "Images avec rotation EXIF": n_exif,
        "Tailles d'image": "variables",
        "Images sans objet": int(df['no_object'].sum()),
    },
    "Après prétraitement": {
        "Taille du vocabulaire": len(vocab_after),
        "Longueur moyenne des légendes (mots)": round(df['caption_len_clean'].mean(), 2),
        "Images en niveaux de gris": modes.get('L', 0),
        "Images avec rotation EXIF": 0,
        "Tailles d'image": "224x224",
        "Images sans objet": int(df['no_object'].sum()),
    },
})
print(comparison)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['caption_len'], bins=30, alpha=0.5, label='avant')
ax.hist(df['caption_len_clean'], bins=30, alpha=0.5, label='après')
ax.set_xlabel("Longueur moyenne des légendes (mots)")
ax.set_title("Longueur des légendes avant/après prétraitement")
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG}/explo2_caption_len.png', dpi=150)
plt.show()

comparison.to_csv(f'{CAP}/explo_comparison.csv')
sync_drive()

## 10. Embeddings image et texte (CLIP)

Encodeur : CLIP ViT-B/32 (512 dimensions), vecteurs normalisés (produit scalaire = similarité cosinus). On encode les images des trois splits et les **légendes nettoyées du train uniquement** (5 par image, soit 50 000) : les légendes de validation et de test ne doivent jamais entrer dans la base. Le contrôle compare la similarité d'une image avec ses propres légendes et avec des légendes d'autres images. Les paramètres sont notés dans `embedding_info.json`.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device :", device)

MODEL = 'openai/clip-vit-base-patch32'
clip = CLIPModel.from_pretrained(MODEL).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL)
BATCH_IMG, BATCH_TXT = 64, 256

# Compatible avec toutes les versions de transformers : sous-modèle + projection
def img_feats(pixel_values):
    out = clip.vision_model(pixel_values=pixel_values)
    return clip.visual_projection(out.pooler_output)

def txt_feats(input_ids, attention_mask):
    out = clip.text_model(input_ids=input_ids, attention_mask=attention_mask)
    return clip.text_projection(out.pooler_output)

def embed_images(split, ids, info):
    out = []
    for s in tqdm(range(0, len(ids), BATCH_IMG), desc=split):
        paths = [f"/content/images_clean/{split}/{info[i]['file_name']}" for i in ids[s:s+BATCH_IMG]]
        imgs = [Image.open(p).convert('RGB') for p in paths]
        inp = processor(images=imgs, return_tensors='pt').to(device)
        with torch.no_grad():
            e = img_feats(inp['pixel_values'])
        out.append((e / e.norm(dim=-1, keepdim=True)).cpu().numpy())
    return np.concatenate(out)

t0 = time.time()
for split, ids, info in SPLIT_INFO:
    E_split = embed_images(split, ids, info)
    np.save(f'{EMB}/emb_{split}.npy', E_split)
    print(split, E_split.shape)

# Légendes du train : 5 par image (légendes nettoyées, train uniquement)
df_idx = df.set_index('image_id')
rows = [(i, k, c) for i in train_ids for k, c in enumerate(df_idx.loc[i, 'captions_clean'][:5])]
texts = [r[2] for r in rows]

tembs = []
for s in tqdm(range(0, len(texts), BATCH_TXT), desc="textes"):
    inp = processor(text=texts[s:s+BATCH_TXT], return_tensors='pt', padding=True,
                    truncation=True, max_length=77).to(device)
    with torch.no_grad():
        e = txt_feats(inp['input_ids'], inp['attention_mask'])
    tembs.append((e / e.norm(dim=-1, keepdim=True)).cpu().numpy())
T_train = np.concatenate(tembs)
np.save(f'{EMB}/emb_text_train.npy', T_train)
json.dump([[int(r[0]), r[1]] for r in rows], open(f'{EMB}/text_index_train.json', 'w'))
print("textes :", T_train.shape)

# Contrôle : une image doit être plus proche de ses propres légendes que de légendes au hasard
Ei = np.load(f'{EMB}/emb_train.npy')[:1000]
Tt = T_train[:5000].reshape(1000, 5, -1)
own  = float((Ei[:, None, :] * Tt).sum(-1).mean())
rand = float((Ei[:, None, :] * np.roll(Tt, 1, axis=0)).sum(-1).mean())
print("similarité image / ses légendes :", round(own, 3), "| légendes au hasard :", round(rand, 3))

info_emb = {
    "modele": MODEL, "dimension": int(T_train.shape[1]),
    "batch_images": BATCH_IMG, "batch_textes": BATCH_TXT,
    "normalisation": "L2 (vecteurs unitaires, produit scalaire = cosinus)",
    "max_tokens_texte": 77,
    "legendes_indexees": "légendes nettoyées du train uniquement (5 par image)",
    "torch": torch.__version__, "transformers": transformers.__version__,
    "duree_secondes": round(time.time() - t0, 1),
}
json.dump(info_emb, open(f'{EMB}/embedding_info.json', 'w'), indent=2)
print(info_emb)
sync_drive()

## 11. Base de connaissances : index FAISS

Deux index exacts (`IndexFlatIP`, similarité cosinus car les vecteurs sont normalisés) : un pour les images du train (10 000 vecteurs) et un pour leurs légendes (50 000). Chaque vecteur est relié à son `image_id` (`image_index_ids.json`, `text_index_train.json`), donc aux légendes et métadonnées de `metadata_clean.json`. Les contrôles d'intégrité (auto-retrouvaille, rappel image → légendes) ne sont pas des scores de performance : les vraies requêtes seront des images de validation et de test. La latence est mesurée avec les 100 images de validation. Les paramètres sont notés dans `index_info.json`.

In [ ]:
import faiss

E_train = np.load(f'{EMB}/emb_train.npy').astype('float32')
T_kb    = np.load(f'{EMB}/emb_text_train.npy').astype('float32')
E_val   = np.load(f'{EMB}/emb_val.npy').astype('float32')
text_index = json.load(open(f'{EMB}/text_index_train.json'))   # ligne -> [image_id, numéro de légende]

def build_flat_ip(X):
    t0 = time.time()
    index = faiss.IndexFlatIP(X.shape[1])
    index.add(X)
    return index, round(time.time() - t0, 3)

idx_img, t_img = build_flat_ip(E_train)
idx_txt, t_txt = build_flat_ip(T_kb)

faiss.write_index(idx_img, f'{IDX}/faiss_image_train.index')
faiss.write_index(idx_txt, f'{IDX}/faiss_text_train.index')
json.dump([int(i) for i in train_ids], open(f'{IDX}/image_index_ids.json', 'w'))   # ligne -> image_id

# Contrôle d'intégrité : chaque image du train se retrouve elle-même en premier
D, I = idx_img.search(E_train[:1000], 1)
print("auto-retrouvée en 1er :", float((I[:, 0] == np.arange(1000)).mean()))

# Contrôle image -> légendes (test d'intégrité sur des images du train)
D, I = idx_txt.search(E_train[:1000], 10)
q_ids = np.array(train_ids[:1000])
txt_img_ids = np.array([r[0] for r in text_index])
for k in (1, 5, 10):
    hit = (txt_img_ids[I[:, :k]] == q_ids[:, None]).any(axis=1).mean()
    print(f"Recall@{k} image -> légendes : {hit:.3f}")

# Latence de recherche (ms par requête)
t0 = time.time()
idx_img.search(E_val, 5)
idx_txt.search(E_val, 5)
lat_ms = round((time.time() - t0) / len(E_val) * 1000, 3)
print("latence :", lat_ms, "ms par requête (image + texte)")

index_info = {
    "bibliotheque": "FAISS " + faiss.__version__,
    "type_index": "IndexFlatIP (recherche exacte, sans compression)",
    "metrique": "produit scalaire sur vecteurs L2-normalisés = similarité cosinus",
    "dimension": int(E_train.shape[1]),
    "index_image": {"vecteurs": int(idx_img.ntotal), "temps_construction_s": t_img,
                    "taille_disque_Mo": round(os.path.getsize(f'{IDX}/faiss_image_train.index')/1e6, 1)},
    "index_texte": {"vecteurs": int(idx_txt.ntotal), "temps_construction_s": t_txt,
                    "taille_disque_Mo": round(os.path.getsize(f'{IDX}/faiss_text_train.index')/1e6, 1)},
    "latence_ms_par_requete": lat_ms,
    "liens": "image_index_ids.json (ligne -> image_id) et embeddings/text_index_train.json (ligne -> image_id, légende) ; légendes et métadonnées dans metadata_clean.json via image_id",
    "contenu": "images et légendes du train uniquement (jamais de val ni de test)",
}
json.dump(index_info, open(f'{IDX}/index_info.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(index_info, indent=2, ensure_ascii=False))
sync_drive()

## 12. Audit de fuite et contrôle qualité

L'audit recharge les fichiers sauvegardés sur Drive (il contrôle donc ce qui sera réellement livré). **Anti-fuite** : ids disjoints entre splits, pas de doublon d'image entre splits, base de connaissances limitée au train, aucune légende de validation ou de test indexée, aucune image de validation ou de test retrouvée pour elle-même. **Qualité** : pas de NaN, vecteurs normalisés, ids uniques, effectifs, alignement des index, 5 légendes par image. Le résultat est sauvegardé dans `audit_report.json`.

In [ ]:
splits = json.load(open(f'{CAP}/splits/splits.json'))
train_ids, val_ids, test_ids = splits['train'], splits['val'], splits['test1000']
df = pd.DataFrame(json.load(open(f'{CAP}/metadata_clean.json')))

E = {s: np.load(f'{EMB}/emb_{s}.npy').astype('float32') for s in ['train', 'val', 'test']}
T = np.load(f'{EMB}/emb_text_train.npy').astype('float32')
text_index = json.load(open(f'{EMB}/text_index_train.json'))
img_ids_idx = json.load(open(f'{IDX}/image_index_ids.json'))
idx_img = faiss.read_index(f'{IDX}/faiss_image_train.index')
idx_txt = faiss.read_index(f'{IDX}/faiss_text_train.index')

audit = {}
def check(name, ok, detail=""):
    audit[name] = {"ok": bool(ok), "detail": detail}
    print("OK    " if ok else "ÉCHEC ", name, "|", detail)

tr, va, te = set(train_ids), set(val_ids), set(test_ids)

# ---------- Audit de fuite ----------
check("ids train/val disjoints", not (tr & va), f"{len(tr & va)} commun(s)")
check("ids train/test disjoints", not (tr & te), f"{len(tr & te)} commun(s)")
check("ids val/test disjoints", not (va & te), f"{len(va & te)} commun(s)")
rep = json.load(open(f'{CAP}/cleaning_report.json'))
check("aucun doublon d'image entre splits (hash MD5)",
      rep.get("duplicates_across_splits", -1) == 0,
      f"{rep.get('duplicates_across_splits')} groupe(s)")
check("index image = train uniquement",
      img_ids_idx == train_ids and idx_img.ntotal == len(train_ids), f"{idx_img.ntotal} vecteurs")
kb_text_ids = {r[0] for r in text_index}
check("index texte : aucune légende de val ou de test",
      kb_text_ids <= tr and not (kb_text_ids & (va | te)),
      f"{len(kb_text_ids)} images, {len(text_index)} légendes")

img_ids_arr = np.array(img_ids_idx)
txt_ids_arr = np.array([r[0] for r in text_index])
for s, ids in [('val', val_ids), ('test', test_ids)]:
    _, I1 = idx_img.search(E[s], 10)
    _, I2 = idx_txt.search(E[s], 10)
    self_hit = sum(int(q in set(img_ids_arr[I1[n]]) or q in set(txt_ids_arr[I2[n]]))
                   for n, q in enumerate(ids))
    check(f"{s} : jamais retrouvé pour lui-même", self_hit == 0, f"{self_hit} cas sur {len(ids)}")

# Information (pas un critère de réussite) : légendes identiques à celles d'autres images
kb_caps = {c.strip().lower() for cs in df[df['split'] == 'train']['captions'] for c in cs[:5]}
for s in ['val', 'test']:
    caps_s = [c.strip().lower() for cs in df[df['split'] == s]['captions'] for c in cs]
    same = sum(c in kb_caps for c in caps_s)
    print(f"info : {same} / {len(caps_s)} légendes de {s} identiques mot pour mot à une légende "
          f"d'une autre image de la base")

# ---------- Contrôle qualité ----------
for s in ['train', 'val', 'test']:
    check(f"embeddings {s} : pas de NaN ni inf", np.isfinite(E[s]).all())
check("embeddings texte : pas de NaN ni inf", np.isfinite(T).all())
check("embeddings normalisés (norme 1)",
      np.allclose(np.linalg.norm(E['train'], axis=1), 1, atol=1e-3)
      and np.allclose(np.linalg.norm(T, axis=1), 1, atol=1e-3))
check("ids uniques dans les métadonnées", df['image_id'].is_unique, f"{len(df)} lignes")
check("effectifs 10 000 / 100 / 1 000",
      (len(train_ids), len(val_ids), len(test_ids)) == (10000, 100, 1000))
check("métadonnées = splits", set(df['image_id']) == tr | va | te)
check("textes alignés avec l'index",
      len(text_index) == T.shape[0] == 5 * len(train_ids), f"{T.shape[0]} vecteurs")
n_caps = df['captions'].apply(len)
check("au moins 5 légendes par image", (n_caps >= 5).all(),
      f"{int((n_caps > 5).sum())} image(s) avec plus de 5 légendes (les 5 premières sont indexées)")
check("légendes nettoyées non vides",
      df['captions_clean'].apply(lambda cs: all(len(c) > 0 for c in cs)).all())

# ---------- Exemple de bout en bout : image de validation -> images et légendes proches ----------
df_idx = df.set_index('image_id')
D1, I1 = idx_img.search(E['val'][0:1], 5)
D2, I2 = idx_txt.search(E['val'][0:1], 5)
print("\nimage val", val_ids[0], "| légendes de référence :", df_idx.loc[val_ids[0], 'captions'][:2])
print("images proches :", [int(img_ids_arr[j]) for j in I1[0]])
for j, d in zip(I2[0], D2[0]):
    iid, k = text_index[j]
    print(round(float(d), 3), "|", df_idx.loc[iid, 'captions_clean'][k])

# ---------- Sauvegarde ----------
n_fail = sum(not v["ok"] for v in audit.values())
json.dump({"controles": audit, "echecs": n_fail},
          open(f'{CAP}/audit_report.json', 'w'), indent=2, ensure_ascii=False)
print("\nTotal :", len(audit), "contrôles,", n_fail, "échec(s)")
sync_drive()

## Fichiers produits

Dossier Drive `captioning` :

- `splits/splits.json` : ids des splits figés
- `cleaning_report.json`, `audit_report.json` : audit des données et contrôle anti-fuite
- `objects.json`, `metadata.json`, `metadata_clean.json`, `detector_info.json` : objets extraits et métadonnées
- `explo1.json`, `explo_comparison.csv`, `figures/` : statistiques des deux explorations
- `embeddings/` : embeddings image et texte, `embedding_info.json`
- `index/` : index FAISS et `index_info.json`

Les fichiers volumineux (embeddings et index) restent sur Drive ; les petits fichiers de documentation sont dans le dossier `data/` du dépôt GitHub.